<a href="https://colab.research.google.com/github/takedatmh/toyama/blob/main/toyama_uni_2026_b_finetuning_ojarumaru_v2_4bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4bit量子化版（T4 GPU対応）

`toyama_uni_2026_b_finetuning_ojarumaru_v2.ipynb` をベースに、**4bit量子化(QLoRA)でGoogle Colab の T4 GPU (16GB) 上で動作する**ように変更したバージョンです。

コード中の変更箇所には `[v2からの変更点]` コメントを入れてあります。

> **Google Drive について**
> このノートブックは**既定では Google Drive に一切書き込みません**。データセットも学習済みアダプタも Colab ランタイム内（`/content`）が既定で、Drive に残したい場合だけ該当行のコメントを外してください。内容を把握しないまま実行した人の Drive 容量を消費しないようにするためです。

## v2からの変更点 一覧

### 1. 4bit量子化を実際に適用（本バージョンの主目的）
v2 は `BitsAndBytesConfig(load_in_8bit=True, ...)` を定義していたが **`from_pretrained` に渡し忘れており**、実際には fp16 でフルロード（約13.5GB）していた。本版では `quantization_config` を渡し、NF4 + double quant で 4bit 化（約4GB）することで T4 の16GBに収めた。

- `bnb_4bit_compute_dtype=torch.float16` … T4(sm_75) は bfloat16 非対応のため fp16 固定
- `device_map={"": 0}` … `"auto"` だと CPU オフロードが発生して極端に遅くなるため GPU0 に固定
- `prepare_model_for_kbit_training()` を追加 … 4bitのままLoRA学習するために必須
- `llm_int8_enable_fp32_cpu_offload` は削除 … 8bit用かつCPUオフロード時のみ有効なフラグで、全レイヤGPU配置では無意味

### 2. EOS を学習させるよう修正（出力が止まらない不具合）
v2 は学習テキストに文末トークンが無く、Llama系トークナイザは既定で `add_eos_token=False` のため、モデルが応答の終端を学習できていなかった。結果、推論時に回答後も `max_new_tokens` まで同じ文を繰り返していた。

- データ作成時に `text` の末尾へ `</s>` を付与
- `labels` を `tokenize_fn` 内で自作し、**パディング部分だけ** `-100` でマスク
- `DataCollatorForLanguageModeling` → `default_data_collator` に変更（前者は `pad_token_id` 一致箇所を一律 `-100` にするため、`pad_token == eos_token` だと学習させたい本物のEOSまで損失計算から外れてしまう）

### 3. Google Drive への書き込みを既定で行わないよう変更
v2 はデータセットを無条件で Drive に書き込んでいた。本版では保存先を既定で `/content`（ランタイム内）にし、Drive に保存する行はコメントアウトして任意選択とした。

また、v2 は `save_strategy="no"` かつ保存処理が無く**学習結果がランタイム終了で消えていた**ため、学習済みLoRAアダプタの保存手順を学習セル末尾に用意した。ただしこれも同じ理由から**既定では実行せず、コメントアウト**してある（アダプタは約40〜80MB）。残したい場合のみコメントを外す。

なお、既定では Drive を使わないため、先頭の `drive.mount` セルは上記の Drive 保存を使う場合にのみ必要。

### 4. Colabバッジのリンク修正
v2 のノートブックを指していたため、本ファイル（`..._v2_4bit.ipynb`）を指すよう修正。

### 5. その他の整理
- `fsspec` を「最新化→ダウングレード」する2セルを、最初からバージョン固定でインストールする1行に統合
- 未使用の `torchao` / `import json` を削除
- `model.config.use_cache` を学習時 `False` / 推論時 `True` に明示設定（警告解消と生成の高速化）
- `prepare_model_for_kbit_training(..., gradient_checkpointing_kwargs={"use_reentrant": False})` を指定し警告を解消
- `tokenizer.pad_token` 未設定時のフォールバックを追加
- 学習セルの `file_path` を `globals().get(...)` による既定値付きに変更し、ランタイム再起動後に学習セル単独でも動くよう修正（データ作成セルを実行済みならそちらの選択を尊重）

In [1]:
!rm -rf ~/.cache/huggingface/datasets
!rm -rf ~/.cache/huggingface/hub

In [2]:
# [v2からの変更点] 本ノートブックは既定では Google Drive を使わないため、このセルは
#   「データセットや学習済みアダプタを Drive に保存したい場合」にのみ実行すればよい。
#   （各保存先はデータ作成セル／学習セルのコメントアウト行で切り替えられる）
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# LLMファインチューニングに必要なライブラリ群
#
# [v2からの変更点] 本ノートブックで一度も使用していない torchao を削除（v2からの残置パッケージ）。
# [v2からの変更点] v2 では「fsspec を -U で最新化 → 次のセルで 2025.3.2 に戻す」という
#                  無駄な往復をしていたため、最初からバージョン固定で 1 回だけ入れるようにした。
#                  （datasets が fsspec<=2026.2.0 を要求するため、最新化すると依存衝突が起きる）
!pip install -q \
  transformers \
  datasets \
  accelerate \
  bitsandbytes \
  peft \
  sentencepiece \
  scipy \
  evaluate \
  huggingface-hub \
  "fsspec==2025.3.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.2 which is incompatible.


#データセット作成 おじゃる丸の巻

In [4]:
import json

# [v2からの変更点] 学習テキストの末尾に EOS トークンを付与する。
#   v2 では text に文末を示すトークンが一切含まれておらず、かつ Llama 系トークナイザは
#   既定で add_eos_token=False のため、モデルが「応答の終わり」を学習できていなかった。
#   その結果、推論時に回答した後も max_new_tokens を使い切るまで喋り続ける不具合が発生していた。
EOS_TOKEN = "</s>"  # ELYZA-japanese-Llama-2-7b (Llama-2系) の EOS トークン

# 基本となるおじゃる丸のセリフデータ
seed_data = [
    {"instruction": "自己紹介をしてください。", "output": "マロはおじゃる丸でおじゃる。よろしく頼むぞよ。"},
    {"instruction": "好きな食べ物は何ですか？", "output": "マロはプリンが大好きでおじゃる！一番の好物ぞよ。"},
    {"instruction": "どこから来たのですか？", "output": "ヘイアンチョウからやってきたでおじゃるよ。"},
    {"instruction": "今日の気分はどうですか？", "output": "今日はとても機嫌が良いでおじゃる。遊ぶぞよ！"},
    {"instruction": "電之助を知っていますか？", "output": "電ボのことかえ？マロの大切なお供でおじゃる。"},
    {"instruction": "何か手伝いましょうか？", "output": "くるしゅうない。マロのためにプリンを持ってくるでおじゃる。"},
    {"instruction": "将来の夢は？", "output": "ずっとのんびり、雅に暮らしたいでおじゃるな。"},
    {"instruction": "走ってください！", "output": "マロは走るのが苦手でおじゃる…。誰かおぶってたもれ。"}
]

# 1000件になるようにデータをループさせて増やす
dataset_items = (seed_data * 125)[:1000]

# データセットの保存先
#
# [v2からの変更点] 既定の保存先を Colab ランタイム内(/content)に変更した。
#   v2 は無条件で Google Drive に書き込んでいたため、内容を把握しないまま実行した場合でも
#   Drive の容量を消費してしまっていた。Drive に残したい場合のみ下の行を有効化する。
#   （その場合は先頭の drive.mount セルの実行が必要。ランタイム内に置いた場合は
#     セッション終了で消えるが、このセルを再実行すればいつでも作り直せる）
file_path = "/content/ojarumaru_dataset.jsonl"
# file_path = "/content/drive/MyDrive/ojarumaru_dataset.jsonl"  # ← Drive に保存したい場合はこちらを有効化

with open(file_path, "w", encoding="utf-8") as f:
    for item in dataset_items:
        # プロンプト形式に整形（末尾に EOS を付けて応答の終端を学習させる）
        prompt = f"以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:\n{item['instruction']}\n\n### 応答:\n{item['output']}{EOS_TOKEN}"
        record = {
            "instruction": item["instruction"],
            "output": item["output"],
            "text": prompt
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"1000件のおじゃる丸データセットを作成し、保存しました: {file_path}")

1000件のおじゃる丸データセットを作成し、保存しました: /content/ojarumaru_dataset.jsonl


#FineTuning LoRA実行

In [6]:
import gc
import torch

# メモリの完全解放
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

# [v2からの変更点] 未使用だった import json を削除。
# [v2からの変更点] labels を上書きしてしまう DataCollatorForLanguageModeling をやめ、
#                  default_data_collator を import するように変更（詳細は後述の tokenize_fn 参照）。
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    default_data_collator,
)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
from huggingface_hub import login
from google.colab import userdata

# --- 2. モデルとトークナイザの準備 ---
login(token=userdata.get('HF_TOKEN'))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")

model_name = "elyza/ELYZA-japanese-Llama-2-7b"

# 4-bit (NF4) 量子化設定
#
# [v2からの変更点] v2 は BitsAndBytesConfig(load_in_8bit=True, ...) を定義していたものの
#                  from_pretrained に渡し忘れており、実際には fp16 フルロード（約13.5GB）だった。
#                  T4(16GB) では学習まで回らないため、4bit(NF4) を実際に適用する。
# [v2からの変更点] compute_dtype は float16 固定。T4(sm_75) は bfloat16 非対応のため bf16 は使えない。
# [v2からの変更点] llm_int8_enable_fp32_cpu_offload は 8bit 用かつ CPU オフロード時のみ効くフラグで、
#                  device_map={"": 0}（全レイヤGPU配置）では無意味なので設定しない。
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# [v2からの変更点] pad_token 未定義のモデルでも padding できるように明示設定（Llama系は未設定のことがある）。
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # 学習時は right padding

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,  # [v2からの変更点] 量子化設定を実際に適用（v2は渡し忘れ）
    device_map={"": 0},              # [v2からの変更点] "auto" は CPU オフロードが発生し激遅になるため GPU0 に固定
    torch_dtype=torch.float16
)

# [v2からの変更点] 量子化モデルを学習可能な状態に準備（LoRA周辺層のfp32化・入力への勾配有効化など）。
#                  4bit のまま学習する QLoRA では必須。
# [v2からの変更点] use_reentrant を明示指定し、gradient checkpointing の UserWarning を解消。
model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# [v2からの変更点] gradient checkpointing と KV キャッシュは併用できないため明示的に無効化（警告解消）。
#                  推論セル側で True に戻している。
model.config.use_cache = False

# --- 3. LoRAの設定 ---
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# --- 4. データセット読み込みと前処理 ---
# [v2からの変更点] データ作成セルを実行していないランタイムでも動くよう、file_path の既定値を用意。
#                  （v2 はデータ作成セルの変数に依存しており、再起動後に単独実行すると NameError になった）
#                  データ作成セルを実行済みの場合は、そちらで選んだ保存先をそのまま使う。
file_path = globals().get("file_path", "/content/ojarumaru_dataset.jsonl")

dataset = load_dataset("json", data_files=file_path, split="train")
dataset_small = dataset  # 既に1000件なのでそのまま使用

# [v2からの変更点] labels を自前で作成し、「パディング部分だけ」を -100 でマスクするよう変更。
#   v2 が使っていた DataCollatorForLanguageModeling は input_ids のうち pad_token_id と一致する
#   位置を一律 -100 にするため、pad_token == eos_token の場合に「学習させたい本物の EOS」まで
#   損失計算から除外されてしまい、EOS を覚えられなかった。
def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tokenized["labels"] = [
        [tok if m == 1 else -100 for tok, m in zip(ids, mask)]
        for ids, mask in zip(tokenized["input_ids"], tokenized["attention_mask"])
    ]
    return tokenized

tokenized_dataset = dataset_small.map(
    tokenize_fn,
    batched=True,
    # [v2からの変更点] default_data_collator は文字列列をテンソル化できないため元の列を削除
    remove_columns=dataset_small.column_names,
)

# --- 5. トレーニング引数の設定と学習開始 ---
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,  # デモ用。本格的に学習させる場合は増やしてください
    fp16=True,           # T4 は bf16 非対応のため fp16
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    # [v2からの変更点] tokenize_fn で作った labels をそのまま渡すため default_data_collator を使用
    data_collator=default_data_collator,
)

trainer.train()

# --- 6. 学習した LoRA アダプタの保存（任意 / 既定では実行しない） ---
#
# [v2からの変更点] v2 は save_strategy="no" かつ保存処理も無く、ランタイム終了で学習結果が
#   完全に消えていたため、保存の手順をここに用意した。
#   ただし内容を把握しないまま実行した場合に Google Drive の容量を消費しないよう、
#   既定では実行しない（コメントアウト）。学習結果を残したい場合だけ下記を有効化する。
#   ※ アダプタのサイズは概ね 40〜80MB 程度。Drive に保存する場合は drive.mount が必要。
#
# adapter_dir = "/content/ojarumaru_lora_4bit"                 # ランタイム内のみ（Drive を使わない）
# adapter_dir = "/content/drive/MyDrive/ojarumaru_lora_4bit"   # Drive に残す場合はこちら
# model.save_pretrained(adapter_dir)
# tokenizer.save_pretrained(adapter_dir)
# print(f"LoRAアダプタを保存しました: {adapter_dir}")

Running on cuda


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

trainable params: 19,988,480 || all params: 6,758,404,096 || trainable%: 0.2958


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss


TrainOutput(global_step=125, training_loss=0.13557928466796876, metrics={'train_runtime': 641.5534, 'train_samples_per_second': 1.559, 'train_steps_per_second': 0.195, 'total_flos': 5089791049728000.0, 'train_loss': 0.13557928466796876, 'epoch': 1.0})

# Fine-Tuning後のモデルを利用して推論(Chat)を実行


In [7]:
from transformers import GenerationConfig

model.eval()

# [v2からの変更点] 学習時に use_cache=False にしているため、推論前に KV キャッシュを有効化して生成を高速化。
model.config.use_cache = True

# おじゃる丸データセットの学習形式に合わせたプロンプト
instruction = "自己紹介をして、好きな食べ物を教えてください。"
prompt = f"以下は、タスクを説明する指示です。要求を適切に満たす応答を書きなさい。\n\n### 指示:\n{instruction}\n\n### 応答:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 推論設定
# [v2からの変更点] 学習データ末尾に EOS を入れた（データ作成セル参照）ことで、
#                  ここでの eos_token_id による停止が実際に機能するようになる。
#                  v2 では EOS を学習していなかったため、回答後も max_new_tokens まで
#                  同じ文を繰り返し続ける出力になっていた。
generation_config = GenerationConfig(
    max_new_tokens=128,
    do_sample=True,
    top_p=0.95,
    temperature=0.7,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id
)

with torch.no_grad():
    output = model.generate(
        **inputs,
        generation_config=generation_config
    )

# 出力をプロンプトと分離して表示
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("回答:\n", generated_text.replace(prompt, "").strip())

回答:
 マロはプリンが大好きでおじゃる！一番の好物ぞよ。
